# Notebook 01 — Paper Ingestion and Benchmark Construction

**Paper:** *Graph-Structured Reinforcement Learning for Scientific Hypothesis Generation in Materials Design*  
**Authors:** Subhadeep Pal, Markus J. Buehler (MIT)  

This notebook implements the full pipeline described in **Section 4.2** of the paper:

```
PDF papers  ──►  Marker (PDF→Markdown)  ──►  gpt-4o-mini (metadata extraction)
            ──►  gpt-5.4 (question generation)  ──►  gpt-5.4 (question refinement)
            ──►  benchmark_questions.json  (100 open-ended questions)
```

**Outputs:**
- `data/corpus/marker_runs/` — per-paper Markdown + JSON extracted fields
- `data/benchmark/benchmark_questions.json` — the 100-question benchmark

**Prerequisites:** `pip install marker-pdf openai python-dotenv pandas`

Set your OpenAI key in a `.env` file at the repo root:
```
OPENAI_API_KEY=sk-...
```

## 1. Imports and Configuration

In [ ]:
import os
import re
import json
from pathlib import Path
from typing import Optional, Dict, Any, List

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

# Load API credentials from .env at the repo root
load_dotenv(Path(__file__).parents[1] / ".env" if "__file__" in dir() else Path("../.env"))
openai_key = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise RuntimeError("OPENAI_API_KEY not found. Add it to a .env file at the repo root.")

client = OpenAI(api_key=openai_key)

# Paths relative to the repo root (one level up from notebooks/)
REPO_ROOT     = Path("..").resolve()
RAW_PDF_DIR   = REPO_ROOT / "data" / "corpus" / "raw_papers"   # put your PDFs here
MARKER_DIR    = REPO_ROOT / "data" / "corpus" / "marker_runs"  # output: one dir per paper
QUESTIONS_OUT = REPO_ROOT / "data" / "benchmark" / "benchmark_questions.json"

MARKER_DIR.mkdir(parents=True, exist_ok=True)


## 2. Step 1 — Convert PDFs to Markdown with Marker

The [Marker](https://github.com/datalab-to/marker) library performs layout-aware text extraction from PDFs without OCR or LLM-assisted parsing (Section 4.2).  
Each PDF produces a Markdown file in `data/corpus/marker_runs/<paper_id>/`.

In [ ]:
from marker.config.parser import ConfigParser
from marker.models import create_model_dict
from marker.converters.pdf import PdfConverter
from marker.output import text_from_rendered

def pdf_to_markdown(pdf_path: Path, out_dir: Path) -> Path:
    """Convert a single PDF to Markdown using Marker.

    Args:
        pdf_path: Path to the input PDF file.
        out_dir:  Directory where paper_id.md and paper_id_meta.json are saved.

    Returns:
        Path to the generated Markdown file.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    out_md = out_dir / (pdf_path.stem + ".md")

    if out_md.exists():
        print(f"  [skip] {pdf_path.name} — already converted")
        return out_md

    config = ConfigParser({})
    model_dict = create_model_dict()
    converter = PdfConverter(config=config.generate_config_dict(), artifact_dict=model_dict)
    rendered = converter(str(pdf_path))
    text, _, _ = text_from_rendered(rendered)

    out_md.write_text(text, encoding="utf-8")
    print(f"  [done] {pdf_path.name} → {out_md.name}")
    return out_md


def convert_all_pdfs(raw_dir: Path = RAW_PDF_DIR, marker_dir: Path = MARKER_DIR):
    """Convert every PDF in raw_dir to Markdown."""
    pdf_files = sorted(raw_dir.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDFs in {raw_dir}")
    for i, pdf_path in enumerate(pdf_files, 1):
        paper_id = f"paper_{i:02d}"
        out_dir  = marker_dir / paper_id
        print(f"Processing [{i}/{len(pdf_files)}]: {pdf_path.name}")
        pdf_to_markdown(pdf_path, out_dir)


# Run conversion (comment out if already done)
# convert_all_pdfs()


## 3. Step 2 — Extract Structured Metadata with gpt-4o-mini

Each Markdown file is passed to `gpt-4o-mini` which extracts structured fields:
`title`, `doi`, `abstract`, `results`, `discussion`, `conclusion`.
The Introduction, Methods, and References sections are intentionally excluded (Section 4.2).

In [ ]:
# JSON schema enforced as a structured output from the model
PAPER_EXTRACT_JSON_SCHEMA: Dict[str, Any] = {
    "name": "paper_section_extraction",
    "schema": {
        "type": "object",
        "properties": {
            "title":       {"type": "string"},
            "doi":         {"type": ["string", "null"]},
            "abstract":    {"type": "string"},
            "results":     {"type": "string"},
            "discussion":  {"type": "string"},
            "conclusion":  {"type": "string"},
        },
        "required": ["title", "abstract", "results", "discussion", "conclusion"],
        "additionalProperties": False,
    },
    "strict": True,
}

SYSTEM_PROMPT = (
    "You are an expert scientific text extractor. "
    "Given raw Markdown text of a research paper, extract the requested sections verbatim. "
    "If a section is missing or merged with another (e.g. 'Results and Discussion'), "
    "split it appropriately. Return an empty string for truly absent sections."
)

def extract_paper_fields(md_text: str) -> Dict[str, Any]:
    """Extract structured fields from paper Markdown using gpt-4o-mini."""
    resp = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": [{"type": "input_text", "text": SYSTEM_PROMPT}]},
            {"role": "user",   "content": [{"type": "input_text", "text": md_text[:30000]}]},
        ],
        text={"format": {"type": "json_schema", "json_schema": PAPER_EXTRACT_JSON_SCHEMA}},
    )
    return json.loads(resp.output_text)


def run_metadata_extraction(marker_dir: Path = MARKER_DIR):
    """Extract metadata for every converted paper and save as *_cleaned.json."""
    paper_dirs = sorted(p for p in marker_dir.iterdir() if p.is_dir())
    print(f"Extracting metadata from {len(paper_dirs)} papers...")

    for paper_dir in tqdm(paper_dirs):
        md_files = list(paper_dir.glob("*.md"))
        if not md_files:
            continue
        md_text = md_files[0].read_text(encoding="utf-8", errors="replace")
        out_json = paper_dir / (md_files[0].stem + "_cleaned.json")

        if out_json.exists():
            continue  # already extracted

        try:
            fields = extract_paper_fields(md_text)
            fields["source_file"] = str(md_files[0].relative_to(REPO_ROOT))
            out_json.write_text(json.dumps(fields, indent=2, ensure_ascii=False), encoding="utf-8")
        except Exception as e:
            print(f"  [error] {paper_dir.name}: {e}")


# Run extraction (comment out if already done)
# run_metadata_extraction()


## 4. Step 3 — Consolidate Paper Corpus

Merge all extracted JSON files into a single `consolidated_papers.json` used as input for question generation.

In [ ]:
def merge_paper_text(record: Dict[str, Any]) -> str:
    """Concatenate abstract + results + discussion + conclusion into one text block."""
    sections = ["abstract", "results", "discussion", "conclusion"]
    chunks = []
    for sec in sections:
        content = (record.get(sec) or "").strip()
        if content:
            chunks.append(f"## {sec.upper()}\n{content}")
    return "\n\n".join(chunks)


def consolidate_corpus(marker_dir: Path = MARKER_DIR,
                       out_json: Path = REPO_ROOT / "data" / "corpus" / "consolidated_papers.json"):
    """Merge all per-paper JSON files into one JSONL corpus file."""
    records = []
    for paper_dir in sorted(marker_dir.iterdir()):
        cleaned_files = list(paper_dir.glob("*_cleaned.json"))
        if not cleaned_files:
            continue
        rec = json.loads(cleaned_files[0].read_text(encoding="utf-8"))
        rec["text"] = merge_paper_text(rec)
        if len(rec["text"]) >= 200:   # skip empty / near-empty extractions
            records.append(rec)

    out_json.write_text(
        "\n".join(json.dumps(r, ensure_ascii=False) for r in records),
        encoding="utf-8"
    )
    print(f"Consolidated {len(records)} papers → {out_json}")
    return records


# Run consolidation (comment out if already done)
# corpus = consolidate_corpus()


## 5. Step 4 — Generate Benchmark Questions with gpt-5.4

For each paper, `gpt-5.4` with `high` reasoning effort generates one self-contained, research-level question covering one of five reasoning categories (Section 4.2):

| Category | Description |
|---|---|
| `causal_multiscale_reasoning` | Multi-scale cause-effect chains |
| `tradeoff_and_non_monotonicity` | Non-obvious competing objectives |
| `hidden_variable_identification` | Latent confounders or mediators |
| `model_abstraction_and_breakdown` | When models fail or simplify |
| `cross_domain_mapping` | Transfer of mechanisms across fields |

In [ ]:
from pydantic import BaseModel, Field

REASONING_CATEGORIES = [
    "causal_multiscale_reasoning",
    "tradeoff_and_non_monotonicity",
    "hidden_variable_identification",
    "model_abstraction_and_breakdown",
    "cross_domain_mapping",
]

class QuestionSet(BaseModel):
    tag:          str = Field(description="2-4 word noun-phrase topic tag")
    category:     str = Field(description="One of the five reasoning categories")
    context:      str = Field(description="Self-contained scientific context (100-150 words)")
    question:     str = Field(description="The reasoning question (begins 'Explain why' or 'Then identify')")
    doi:          Optional[str] = Field(default=None, description="DOI of the source paper")

GENERATION_SYSTEM = (
    "You are a rigorous scientific examiner designing PhD journal-club level evaluation questions. "
    "Each question must: define a clear scientific system, include relevant variables and mechanisms, "
    "pose a non-trivial reasoning challenge involving a tradeoff, hidden variable, failure mode, "
    "or mechanistic breakdown. Format: single paragraph, 150-200 words, self-contained."
)

def generate_question(paper_text: str, doi: Optional[str] = None) -> Optional[Dict[str, Any]]:
    """Generate one evaluation question from a paper's text using gpt-5.4."""
    prompt = (
        f"Generate one research-level evaluation question from the following paper text.\n\n"
        f"Paper text:\n{paper_text[:8000]}\n\n"
        f"Assign it to one of these categories: {', '.join(REASONING_CATEGORIES)}"
    )
    try:
        resp = client.responses.create(
            model="gpt-5.4",
            input=[
                {"role": "system", "content": GENERATION_SYSTEM},
                {"role": "user",   "content": prompt},
            ],
            reasoning={"effort": "high"},
            text={"format": {"type": "json_object"}},
        )
        result = json.loads(resp.output_text)
        result["doi"] = doi
        return result
    except Exception as e:
        print(f"  [error] question generation: {e}")
        return None


def build_benchmark(corpus_path: Path = REPO_ROOT / "data" / "corpus" / "consolidated_papers.json",
                    out_path: Path = QUESTIONS_OUT):
    """Generate one question per paper and save to benchmark_questions.json."""
    records = [json.loads(l) for l in corpus_path.read_text().splitlines() if l.strip()]
    questions = []

    for i, rec in enumerate(tqdm(records, desc="Generating questions")):
        q = generate_question(rec["text"], doi=rec.get("doi"))
        if q:
            q["id"]    = i
            q["title"] = rec.get("title", "")
            questions.append(q)

    out_path.write_text(json.dumps(questions, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Benchmark saved: {len(questions)} questions → {out_path}")
    return questions


# Run generation (comment out if already done — API calls are expensive)
# questions = build_benchmark()


## 6. Step 5 — Refine Questions with gpt-5.4

A second gpt-5.4 pass improves readability, grammar, and benchmark suitability while preserving the original causal structure and reasoning challenge (Section 4.2).

In [ ]:
REFINEMENT_SYSTEM = (
    "You are a scientific writing editor. Improve the clarity, grammar, and precision of the "
    "provided question WITHOUT changing: the scientific system, variables, causal structure, "
    "question type, or intended reasoning challenge. Do NOT simplify the task or introduce "
    "new scientific claims."
)

def refine_question(question_text: str) -> Optional[str]:
    """Polish a question for readability using gpt-5.4 with medium reasoning effort."""
    try:
        resp = client.responses.create(
            model="gpt-5.4",
            input=[
                {"role": "system", "content": REFINEMENT_SYSTEM},
                {"role": "user",   "content": f"Refine this question:\n\n{question_text}"},
            ],
            reasoning={"effort": "medium"},
        )
        return resp.output_text.strip()
    except Exception as e:
        print(f"  [error] refinement: {e}")
        return question_text  # fallback: keep original


def refine_benchmark(questions_path: Path = QUESTIONS_OUT):
    """Refine all questions in-place."""
    questions = json.loads(questions_path.read_text())
    for q in tqdm(questions, desc="Refining questions"):
        refined = refine_question(q.get("question", ""))
        if refined:
            q["question"] = refined

    questions_path.write_text(json.dumps(questions, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Refinement complete. {len(questions)} questions saved.")


# Run refinement (comment out if already done)
# refine_benchmark()


## 7. Inspect the Final Benchmark

The benchmark `data/benchmark/benchmark_questions.json` contains 100 open-ended questions. Each entry has keys: `id`, `tag`, `category`, `context`, `question`, `doi`, `title`.

In [ ]:
import pandas as pd

# Load and preview the benchmark
benchmark = pd.read_json(QUESTIONS_OUT)
print(f"Benchmark size: {len(benchmark)} questions")
print("\nCategory distribution:")
print(benchmark["category"].value_counts())
print("\nSample question:")
sample = benchmark.iloc[0]
print(f"  Tag:      {sample['tag']}")
print(f"  Category: {sample['category']}")
print(f"  Context:  {sample['context'][:200]}...")
print(f"  Question: {sample['question'][:200]}...")
